# Обучение Student-модели

Notebook для обучения компактной heatmap-based Student-модели на COCO-like pseudo-labeled dataset.

In [ ]:
!rm -rf /kaggle/working/pose-estimation
!git clone https://github.com/abbos-trnv/pose-estimation.git /kaggle/working/pose-estimation
%cd /kaggle/working/pose-estimation

In [ ]:
from pathlib import Path

# Замените путь, если Kaggle Dataset называется иначе.
COCO_DATA_DIR = Path('/kaggle/input/nuscenes-pose-coco/nuscenes_pose_coco')
OUTPUT_DIR = Path('/kaggle/working/models/student_baseline')

!ls -lah {COCO_DATA_DIR}
!cat {COCO_DATA_DIR / 'report.json'}

In [ ]:
!pip install -q tqdm opencv-python

In [ ]:
# Первый запуск лучше сделать коротким, чтобы проверить pipeline.
!python scripts/train_student.py \
  --config configs/training/student_baseline.json \
  --data-dir {COCO_DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --epochs 3

In [ ]:
!ls -lah {OUTPUT_DIR}
!cat {OUTPUT_DIR / 'history.json'}

In [ ]:
STUDENT_QA_DIR = Path('/kaggle/working/data/qa/student_predictions')

!python scripts/visualize_student_predictions.py \
  --data-dir {COCO_DATA_DIR} \
  --checkpoint {OUTPUT_DIR / 'best.pt'} \
  --output-dir {STUDENT_QA_DIR} \
  --num-samples 64

from IPython.display import Image, display
display(Image(filename=str(STUDENT_QA_DIR / 'student_predictions_grid.jpg')))

In [ ]:
!python scripts/benchmark_student_speed.py \
  --checkpoint {OUTPUT_DIR / 'best.pt'} \
  --output-path /kaggle/working/logs/student_speed_b1.json \
  --batch-size 1

!python scripts/benchmark_student_speed.py \
  --checkpoint {OUTPUT_DIR / 'best.pt'} \
  --output-path /kaggle/working/logs/student_speed_b32.json \
  --batch-size 32